In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.4 MB/s eta 0:00:00


In [ ]:
# Импортируем необходимые библиотеки
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Для визуализаций
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Для машинного обучения
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Для кластеризации и классификации
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Устанавливаем seed для воспроизводимости
np.random.seed(42)

In [ ]:
# Загружаем три CSV-файла
glassdoor_jobs = pd.read_csv('glassdoor_jobs_2024.csv', on_bad_lines='skip')
glassdoor_salaries = pd.read_csv('Glassdoor_Salary_Cleaned_Version_2024.csv', on_bad_lines='skip')
ds_salaries = pd.read_csv('ds_salaries_2023.csv', header=None, on_bad_lines='skip')

# Присваиваем имена колонкам для ds_salaries (известная структура Kaggle-датасета)
ds_salaries.columns = [
    'work_year', 'experience_level', 'employment_type', 'job_title',
    'salary', 'salary_currency', 'salary_in_usd', 'employee_residence',
    'remote_ratio', 'company_location', 'company_size'
]

# Оставляем только зарплаты в USD и за 2023 год (актуальность)
ds_salaries = ds_salaries[
    (ds_salaries['salary_currency'] == 'USD') &
    (ds_salaries['work_year'] == 2023)
].copy()

# Преобразуем зарплату в тыс. USD для единообразия
ds_salaries['avg_salary'] = ds_salaries['salary_in_usd'] / 1000.0

# Выбираем нужные колонки
ds_salaries = ds_salaries[['job_title', 'avg_salary', 'company_location', 'company_size']]
ds_salaries.rename(columns={'company_location': 'state'}, inplace=True)

In [ ]:
# Удаляем строки без salary_estimate
df_g = glassdoor_salaries.copy()
df_g = df_g[df_g.iloc[:, 1].str.contains(r'\$.*K', na=False)].copy()  # предполагаем, что salary_estimate — 2-й столбец

# Извлекаем название вакансии и salary_estimate
df_g['job_title'] = df_g.iloc[:, 0]
df_g['salary_estimate'] = df_g.iloc[:, 1]

# Парсим зарплату: "$86K-$141K" → min=86, max=141
def parse_salary(s):
    match = re.search(r'\$(\d+)K-\$(\d+)K', s)
    if match:
        return float(match.group(1)), float(match.group(2))
    return np.nan, np.nan

df_g['min_k'], df_g['max_k'] = zip(*df_g['salary_estimate'].apply(parse_salary))
df_g = df_g.dropna(subset=['min_k', 'max_k']).copy()
df_g['avg_salary'] = (df_g['min_k'] + df_g['max_k']) / 2

# Извлекаем признаки навыков (последние 7 столбцов — бинарные флаги)
skill_cols = ['python', 'r', 'sql', 'tableau', 'aws', 'spark', 'hadoop']
df_g[skill_cols] = df_g.iloc[:, -7:].values  # последние 7 колонок

# Извлекаем штат (предположим, что он в колонке с индексом -9)
df_g['state'] = df_g.iloc[:, -9].astype(str).str.split(',').str[-1].str.strip()

# Удаляем лишние колонки
df_g = df_g[['job_title', 'avg_salary', 'state'] + skill_cols].copy()

In [ ]:
# Объединяем Glassdoor и ds_salaries
df = pd.concat([df_g, ds_salaries], ignore_index=True)

# Создаём признак уровня опыта по названию вакансии
def infer_experience(title):
    title = str(title).lower()
    if any(word in title for word in ['senior', 'lead', 'principal', 'staff', 'director', 'head']):
        return 'Senior'
    elif any(word in title for word in ['junior', 'associate', 'intern', 'i ', 'ii ']):
        return 'Junior'
    else:
        return 'Mid'

df['experience_level'] = df['job_title'].apply(infer_experience)

# Удаляем строки без зарплаты или должности
df = df.dropna(subset=['job_title', 'avg_salary']).copy()

In [ ]:
df.to_csv('salaries_01.csv')

In [ ]:
df.head()

,job_title,avg_salary,state,python,r,sql,tableau,aws,spark,hadoop,company_size,experience_level
0,Data Scientist,72.0,Data Scientist,0,0,1,Data Scientist,$53K-$91K (Glassdoor est.),53.0,91.0,NaN,Mid
1,Healthcare Data Scientist,87.5,Healthcare Data Scientist,0,0,0,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),63.0,112.0,NaN,Mid
2,Data Scientist,85.0,Data Scientist,1,0,1,Data Scientist,$80K-$90K (Glassdoor est.),80.0,90.0,NaN,Mid
3,Data Scientist,76.5,Data Scientist,0,0,0,Data Scientist,$56K-$97K (Glassdoor est.),56.0,97.0,NaN,Mid
4,Data Scientist,114.5,Data Scientist,0,0,1,Data Scientist,$86K-$143K (Glassdoor est.),86.0,143.0,NaN,Mid


In [ ]:
# Группируем по job_title
abc_xyz = df.groupby('job_title').agg(
    total_salary=('avg_salary', 'sum'),
    count=('avg_salary', 'count'),
    mean_salary=('avg_salary', 'mean'),
    std_salary=('avg_salary', 'std')
).fillna(0)

# Коэффициент вариации
abc_xyz['cv'] = abc_xyz['std_salary'] / abc_xyz['mean_salary']
abc_xyz['cv'] = abc_xyz['cv'].replace([np.inf, -np.inf], 0)

# ABC: по накопленной доле общей суммы
abc_xyz = abc_xyz.sort_values('total_salary', ascending=False)
abc_xyz['cumsum_pct'] = abc_xyz['total_salary'].cumsum() / abc_xyz['total_salary'].sum()

abc_xyz['abc_class'] = abc_xyz['cumsum_pct'].apply(
    lambda x: 'A' if x <= 0.8 else ('B' if x <= 0.95 else 'C')
)

# XYZ: по коэффициенту вариации
abc_xyz['xyz_class'] = abc_xyz['cv'].apply(
    lambda x: 'X' if x < 0.25 else ('Y' if x < 0.5 else 'Z')
)

# Добавляем обратно в основной датасет
df = df.merge(abc_xyz[['abc_class', 'xyz_class']], on='job_title', how='left')

In [ ]:
df.to_csv('salaries_02.csv')

In [ ]:
# FM-анализ (частота-доход)
fm = df.groupby('job_title').agg(
    frequency=('job_title', 'count'),
    monetary=('avg_salary', 'mean')
).reset_index()

# Удаляем строки с NaN в frequency или monetary (на всякий случай)
fm = fm.dropna(subset=['frequency', 'monetary'])

# Нормализуем от 1 до 5 с обработкой дубликатов
try:
    fm['F_score'] = pd.qcut(fm['frequency'], 5, labels=[1,2,3,4,5], duplicates='drop').astype(int)
except ValueError:
    # Если qcut всё равно не справляется — используем rank + cut
    fm['F_score'] = pd.cut(fm['frequency'].rank(method='first'), bins=5, labels=[1,2,3,4,5]).astype(int)

try:
    fm['M_score'] = pd.qcut(fm['monetary'], 5, labels=[1,2,3,4,5], duplicates='drop').astype(int)
except ValueError:
    fm['M_score'] = pd.cut(fm['monetary'].rank(method='first'), bins=5, labels=[1,2,3,4,5]).astype(int)

# Сегменты
def fm_segment(row):
    if row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champion'
    elif row['F_score'] <= 2 and row['M_score'] >= 4:
        return 'Niche'
    elif row['F_score'] >= 4 and row['M_score'] <= 2:
        return 'Common'
    else:
        return 'Emerging'

fm['fm_segment'] = fm.apply(fm_segment, axis=1)

# Добавляем обратно в основной датасет
df = df.merge(fm[['job_title', 'fm_segment']], on='job_title', how='left')

In [ ]:
df.to_csv('salaries_03.csv')

In [ ]:
# Выбираем признаки
features = ['job_title', 'state', 'experience_level'] + skill_cols
X = df[features].copy()
y = df['avg_salary'].copy()

# Разделение
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor: One-Hot для категориальных, passthrough для бинарных навыков
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['job_title', 'state', 'experience_level']),
        ('num', 'passthrough', skill_cols)
    ]
)

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 2.7 MB/s eta 0:00:00


In [ ]:
import re

# Загружаем файл
df = pd.read_csv('salaries_03.csv')

# Определяем, какой столбец содержит строку с зарплатой
# Судя по данным, это столбец с индексом 8 (или именем 'aws' из-за смещения)
# Проверим первые значения
print("Примеры значений в столбце 'aws':")
print(df['aws'].head(5))

# Функция для извлечения min и max из строки вида "$53K-$91K" или "Employer Provided Salary:$150K-$160K"
def extract_salary_range(s):
    if pd.isna(s):
        return pd.NA, pd.NA
    # Ищем шаблон: $числоK-$числоK (цифры до и после дефиса)
    match = re.search(r'\$(\d+)K.*?\$(\d+)K', str(s))
    if match:
        min_val = float(match.group(1))
        max_val = float(match.group(2))
        return min_val, max_val
    else:
        # Если не найдено — возвращаем NaN
        return pd.NA, pd.NA

# Применяем функцию
df['min_salary'], df['max_salary'] = zip(*df['aws'].apply(extract_salary_range))

# Преобразуем в числовой тип
df['min_salary'] = pd.to_numeric(df['min_salary'], errors='coerce')
df['max_salary'] = pd.to_numeric(df['max_salary'], errors='coerce')

# Проверяем результат
print("\nРезультат после извлечения:")
print(df[['aws', 'min_salary', 'max_salary']].head(10))

# Сохраняем обновлённый файл (опционально)
df.to_csv('salaries_03_with_min_max.csv', index=False)
print("\n✅ Файл сохранён как 'salaries_03_with_min_max.csv'")

Примеры значений в столбце 'aws':
0     $53K-$91K (Glassdoor est.)
1    $63K-$112K (Glassdoor est.)
2     $80K-$90K (Glassdoor est.)
3     $56K-$97K (Glassdoor est.)
4    $86K-$143K (Glassdoor est.)
Name: aws, dtype: object

Результат после извлечения:
                            aws  min_salary  max_salary
0    $53K-$91K (Glassdoor est.)        53.0        91.0
1   $63K-$112K (Glassdoor est.)        63.0       112.0
2    $80K-$90K (Glassdoor est.)        80.0        90.0
3    $56K-$97K (Glassdoor est.)        56.0        97.0
4   $86K-$143K (Glassdoor est.)        86.0       143.0
5   $71K-$119K (Glassdoor est.)        71.0       119.0
6    $54K-$93K (Glassdoor est.)        54.0        93.0
7   $86K-$142K (Glassdoor est.)        86.0       142.0
8    $38K-$84K (Glassdoor est.)        38.0        84.0
9  $120K-$160K (Glassdoor est.)       120.0       160.0

✅ Файл сохранён как 'salaries_03_with_min_max.csv'


In [ ]:

# 3. Загрузка данных
df = pd.read_csv('/content/salaries_03_with_min_max.csv')

# 4. Проверка структуры и типов данных
print("Первые 5 строк:")
print(df.head())
print("\nТипы данных:")
print(df.dtypes)



Первые 5 строк:
   Unnamed: 0                  job_title  avg_salary  \
0           0             Data Scientist        72.0   
1           1  Healthcare Data Scientist        87.5   
2           2             Data Scientist        85.0   
3           3             Data Scientist        76.5   
4           4             Data Scientist       114.5   

                       state  python  r  sql                    tableau  \
0             Data Scientist       0  0    1             Data Scientist   
1  Healthcare Data Scientist       0  0    0  Healthcare Data Scientist   
2             Data Scientist       1  0    1             Data Scientist   
3             Data Scientist       0  0    0             Data Scientist   
4             Data Scientist       0  0    1             Data Scientist   

                           aws  spark  hadoop  company_size experience_level  \
0   $53K-$91K (Glassdoor est.)   53.0    91.0           NaN              Mid   
1  $63K-$112K (Glassdoor est.)   63.

In [ ]:
import category_encoders as ce
# 5. Убедимся, что min_salary и max_salary числовые
df['min_salary'] = pd.to_numeric(df['min_salary'], errors='coerce')
df['max_salary'] = pd.to_numeric(df['max_salary'], errors='coerce')

# Удалим строки, где зарплаты не определены
df = df.dropna(subset=['min_salary', 'max_salary']).copy()

# 6. Определим признаки и целевые переменные
# Целевые переменные:
y_min = df['min_salary']
y_max = df['max_salary']

# Признаки (все столбцы, кроме целевых и тех, что не нужны)
# Уберём: индекс, job_title (дублирует state в этом файле), avg_salary, salary_estimate
feature_cols = ['state', 'python', 'r', 'sql', 'tableau', 'aws', 'spark', 'hadoop', 'company_size', 'experience_level']

X = df[feature_cols].copy()

print("\nПризнаки для моделирования:")
print(X.head())
print("\nТипы признаков:")
print(X.dtypes)

# 7. Определим категориальные и числовые колонки
categorical_cols = ['state', 'company_size', 'experience_level']
numerical_cols = ['python', 'r', 'sql', 'tableau', 'aws', 'spark', 'hadoop']

# Убедимся, что числовые колонки — действительно числовые
for col in numerical_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0).astype(int)

# 8. Кодирование категориальных признаков с помощью TargetEncoder
# TargetEncoder использует среднее значение целевой переменной для кодирования
# Так как у нас две целевые переменные — обучим два энкодера или используем одну (например, avg_salary)

# Создаём временную целевую переменную для кодирования
y_avg_for_encoding = (y_min + y_max) / 2

# Инициализируем TargetEncoder
encoder = ce.TargetEncoder(cols=categorical_cols)

# Подгоняем энкодер на обучающих данных (всё X и y_avg_for_encoding)
X_encoded = encoder.fit_transform(X, y_avg_for_encoding)

# 9. Проверка результата
print("\nПосле кодирования (первые 5 строк):")
print(X_encoded.head())
print("\nТипы после кодирования:")
print(X_encoded.dtypes)

# 10. Готово! Теперь:
# X_encoded — числовая матрица признаков
# y_min, y_max — целевые переменные

# Сохраняем для дальнейшего использования (опционально)
X_encoded.to_csv('X_encoded_for_ml.csv', index=False)
y_min.to_csv('y_min.csv', index=False)
y_max.to_csv('y_max.csv', index=False)

print("\n✅ Данные подготовлены и сохранены!")


Признаки для моделирования:
                       state  python  r  sql                    tableau  \
0             Data Scientist       0  0    1             Data Scientist   
1  Healthcare Data Scientist       0  0    0  Healthcare Data Scientist   
2             Data Scientist       1  0    1             Data Scientist   
3             Data Scientist       0  0    0             Data Scientist   
4             Data Scientist       0  0    1             Data Scientist   

                           aws  spark  hadoop  company_size experience_level  
0   $53K-$91K (Glassdoor est.)   53.0    91.0           NaN              Mid  
1  $63K-$112K (Glassdoor est.)   63.0   112.0           NaN              Mid  
2   $80K-$90K (Glassdoor est.)   80.0    90.0           NaN              Mid  
3   $56K-$97K (Glassdoor est.)   56.0    97.0           NaN              Mid  
4  $86K-$143K (Glassdoor est.)   86.0   143.0           NaN              Mid  

Типы признаков:
state                object
p

In [ ]:
# 2. Импорт
import pandas as pd
import numpy as np
import re
import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 3. Загрузка данных
df = pd.read_csv('salaries_03.csv')

# 4. Парсинг min_salary и max_salary из столбца 'aws'
def extract_salary_range(s):
    if pd.isna(s):
        return pd.NA, pd.NA
    # Ищем шаблон: $числоK-$числоK
    match = re.search(r'\$(\d+)K.*?\$(\d+)K', str(s))
    if match:
        min_val = float(match.group(1))
        max_val = float(match.group(2))
        return min_val, max_val
    else:
        return pd.NA, pd.NA

# Применяем функцию
df['min_salary'], df['max_salary'] = zip(*df['aws'].apply(extract_salary_range))

# Преобразуем в числовой тип
df['min_salary'] = pd.to_numeric(df['min_salary'], errors='coerce')
df['max_salary'] = pd.to_numeric(df['max_salary'], errors='coerce')

# Удаляем строки без зарплат
df = df.dropna(subset=['min_salary', 'max_salary']).copy()

# 5. Определяем признаки и целевые переменные
feature_cols = ['state', 'company_size', 'experience_level', 'python', 'r', 'sql', 'tableau', 'aws', 'spark', 'hadoop']
X = df[feature_cols].copy()
y_min = df['min_salary']
y_max = df['max_salary']

# 6. Разделение на обучающую и тестовую выборки
X_train, X_test, y_train_min, y_test_min = train_test_split(X, y_min, test_size=0.2, random_state=42)
_, _, y_train_max, y_test_max = train_test_split(X, y_max, test_size=0.2, random_state=42)

# 7. Кодирование категориальных признаков СРАЗУ ПОСЛЕ РАЗДЕЛЕНИЯ
cat_cols = ['state', 'company_size', 'experience_level']

# Создаём TargetEncoder
encoder = ce.TargetEncoder(cols=cat_cols)

# Подгоняем энкодер на обучающих данных
X_train_encoded = encoder.fit_transform(X_train, y_train_min)  # используем y_train_min как прокси-целевую переменную
X_test_encoded = encoder.transform(X_test)  # применяем к тестовым данным

In [ ]:
# 8. Определение моделей
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1, max_iter=5000),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'CatBoost': CatBoostRegressor(iterations=100, verbose=0, random_seed=42)
}

results = []

for name, model in models.items():
    print(f"\nОбучение модели: {name}")

    # Обучаем на закодированных данных
    model_min = model.fit(X_train_encoded, y_train_min)
    model_max = model.fit(X_train_encoded, y_train_max)

    # Предсказываем
    y_pred_min = model_min.predict(X_test_encoded)
    y_pred_max = model_max.predict(X_test_encoded)

    # Метрики
    mae_min = mean_absolute_error(y_test_min, y_pred_min)
    rmse_min = np.sqrt(mean_squared_error(y_test_min, y_pred_min))
    r2_min = r2_score(y_test_min, y_pred_min)

    mae_max = mean_absolute_error(y_test_max, y_pred_max)
    rmse_max = np.sqrt(mean_squared_error(y_test_max, y_pred_max))
    r2_max = r2_score(y_test_max, y_pred_max)

    # Среднее по двум целевым переменным
    mae_avg = (mae_min + mae_max) / 2
    rmse_avg = (rmse_min + rmse_max) / 2
    r2_avg = (r2_min + r2_max) / 2

    results.append({
        'Model': name,
        'MAE_min': round(mae_min, 2),
        'MAE_max': round(mae_max, 2),
        'MAE_avg': round(mae_avg, 2),
        'RMSE_min': round(rmse_min, 2),
        'RMSE_max': round(rmse_max, 2),
        'RMSE_avg': round(rmse_avg, 2),
        'R2_min': round(r2_min, 4),
        'R2_max': round(r2_max, 4),
        'R2_avg': round(r2_avg, 4)
    })

# 9. Вывод результатов
results_df = pd.DataFrame(results).sort_values('MAE_avg')
print("\n✅ Сравнение моделей (среднее по min_salary и max_salary):")
print(results_df[['Model', 'MAE_avg', 'RMSE_avg', 'R2_avg']])


Обучение модели: Linear Regression


ValueError: could not convert string to float: 'Data Scientist'